[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-decision-tree.ipynb)

# Decision Trees

*AIBits Academy · Machine Learning End To End · Tree-Based Models*

A hierarchical, interpretable model that makes predictions by learning a sequence of if-else rules derived from the training data.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> Every diagnosis, every fraud call, every loan decision you've ever seen a human expert make is really just a chain of yes/no questions: *Is the income above ₹40 lakh? If yes, is the CIBIL score above 700? If yes, approve.* A decision tree learns exactly that chain of questions directly from the data — and unlike almost every other model in this course, you can literally read the resulting rules out loud in plain language. The trick is that the tree doesn't guess which question to ask first: at every node it exhaustively evaluates every feature at every threshold and picks the one that best separates the two classes. That single "best split" idea, applied recursively, is the entire algorithm.

> **📋 Real-World Case Study — Transaction Fraud Detection**
>
> Payment fraud teams favour decision trees precisely because of their interpretability requirement: when a transaction is auto-declined, compliance and customer support need a traceable explanation, not a black-box score. A tree that splits on "amount > ₹40,000 AND merchant_category = electronics AND time = late_night" gives an auditable, plain-language reason for the decline — a genuine business requirement in regulated financial contexts that pushes teams toward trees (or tree ensembles with SHAP explanations) over harder-to-explain alternatives.

## Tree Structure

A decision tree partitions the feature space recursively. Each internal node tests a feature; each branch represents a threshold split; each leaf gives a prediction (class label or mean value). Trees are fully interpretable — you can trace *exactly* why a prediction was made.

## Splitting Criteria

| Criterion | Formula | Used for |
|---|---|---|
| **Gini Impurity** | 1 − Σ pᵢ² | Classification (CART) |
| **Entropy** | −Σ pᵢ log₂(pᵢ) | Classification (ID3, C4.5) |
| **Information Gain** | H(parent) − Σ wᵢH(child) | Classification |
| **MSE / Variance reduction** | Var(y) before − Σ wᵢVar(yᵢ) | Regression |

## Worked Numeric Example — Computing a Gini Split by Hand

A parent node has 10 HDFC loan applicants: 6 approved, 4 rejected. Splitting on "Income ≥ ₹40L?" sends 5 applicants left (4 approved, 1 rejected) and 5 right (2 approved, 3 rejected):

$$\begin{gathered}\text{Gini(parent)} = 1-\left(\tfrac{6}{10}\right)^2-\left(\tfrac{4}{10}\right)^2 = 1-0.36-0.16 = \mathbf{0.48}\\[4pt]\text{Gini(left)} = 1-\left(\tfrac{4}{5}\right)^2-\left(\tfrac{1}{5}\right)^2 = 1-0.64-0.04 = 0.32\\[4pt]\text{Gini(right)} = 1-\left(\tfrac{2}{5}\right)^2-\left(\tfrac{3}{5}\right)^2 = 1-0.16-0.36 = 0.48\\[4pt]\text{Weighted child Gini} = \tfrac{5}{10}(0.32) + \tfrac{5}{10}(0.48) = \mathbf{0.40}\\[4pt]\text{Gini Gain} = 0.48-0.40 = \mathbf{0.08}\end{gathered}$$

The CART algorithm (implemented in the `_best_split` method below) evaluates this Gini Gain for *every* feature at *every* candidate threshold, and greedily picks whichever split maximises it — exactly the loop you can trace in the from-scratch code.

## Synchronised View — Feature Space ↔ Tree Structure

The animation below runs the two views side-by-side on a real (sklearn-verified, depth-2, 24-applicant) HDFC loan tree. On the left, each greedy split appears in the 2D feature space; on the right, the corresponding tree node grows in step with it. Watch how each cut in the feature space becomes exactly one internal node in the tree — the tree diagram *is* just a hierarchical index over the rectangular partitions on the scatter, not a separate model.

## From Scratch — CART Algorithm

In [ ]:
# Decision Tree CART from scratch
import numpy as np
from collections import Counter

class Node:
    def __init__(self, feat=None, thr=None, left=None, right=None, val=None):
        self.feat=feat; self.thr=thr; self.left=left; self.right=right; self.val=val

class DecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth=max_depth; self.min_samples_split=min_samples_split

    def _gini(self, y):
        n=len(y); counts=Counter(y)
        return 1 - sum((c/n)**2 for c in counts.values())

    def _best_split(self, X, y):
        best_gain, best_feat, best_thr = -np.inf, None, None
        g_parent = self._gini(y)
        for feat in range(X.shape[1]):
            thresholds = np.unique(X[:, feat])
            for thr in thresholds:
                l, r = y[X[:,feat]<=thr], y[X[:,feat]>thr]
                if len(l)==0 or len(r)==0: continue
                gain = g_parent - (len(l)/len(y))*self._gini(l) - (len(r)/len(y))*self._gini(r)
                if gain > best_gain: best_gain,best_feat,best_thr = gain,feat,thr
        return best_feat, best_thr

    def _build(self, X, y, depth=0):
        if (len(set(y))==1 or len(y)or
            (self.max_depth and depth>=self.max_depth)):
            return Node(val=Counter(y).most_common(1)[0][0])
        feat, thr = self._best_split(X, y)
        if feat is None: return Node(val=Counter(y).most_common(1)[0][0])
        mask = X[:,feat] <= thr
        return Node(feat=feat, thr=thr,
                    left=self._build(X[mask], y[mask], depth+1),
                    right=self._build(X[~mask], y[~mask], depth+1))

    def fit(self, X, y): self.root_=self._build(np.array(X), np.array(y))

    def _predict_one(self, x, node):
        if node.val is not None: return node.val
        return (self._predict_one(x, node.left) if x[node.feat]<=node.thr
                else self._predict_one(x, node.right))

    def predict(self, X): return np.array([self._predict_one(x, self.root_) for x in X])

# Test on HDFC loan data
np.random.seed(42)
income=np.random.uniform(15,75,200); cibil=np.random.uniform(540,790,200)
y=(((income>40)&(cibil>660))|((income>55))).astype(int)
X=np.column_stack([income,cibil])
from sklearn.model_selection import train_test_split
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=0)
dt=DecisionTreeClassifier(max_depth=4); dt.fit(X_tr,y_tr)
print(f"Scratch DT accuracy: {np.mean(dt.predict(X_te)==y_te):.3f}")
from sklearn.tree import DecisionTreeClassifier as SKDT
skdt=SKDT(max_depth=4); skdt.fit(X_tr,y_tr)
print(f"sklearn  DT accuracy: {skdt.score(X_te,y_te):.3f}")

## Pruning — Fighting Overfitting

| Strategy | Mechanism | Key parameter |
|---|---|---|
| Pre-pruning (Early stopping) | Limit growth: max_depth, min_samples_leaf, min_impurity_decrease | max_depth (start: 3–8) |
| Post-pruning (CCP) | Grow full tree, then prune branches with ccp_alpha | ccp_alpha (tune via CV) |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Gini impurity

Write `gini(labels)` = `1 - sum(p_k^2)` over the class proportions.

In [ ]:
import numpy as np
def gini(labels):
    pass   # TODO


In [ ]:
try:
    check("pure node is 0", gini([1, 1, 1]) == 0)
    check("50/50 is 0.5", gini([0, 0, 1, 1]) == 0.5)
    check("three classes", round(gini([0, 1, 2]), 4) == 0.6667)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return 1 - (p ** 2).sum()

```

</details>

### Exercise 2 · Medium · A shallow tree on iris

Fit `DecisionTreeClassifier(max_depth=2, random_state=0)` on the iris training split. Store the test accuracy in `acc` and the number of leaves in `n_leaves`.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0, stratify=y)
acc = n_leaves = None   # TODO


In [ ]:
try:
    check("accuracy above 0.9", acc > 0.9)
    check("at most 4 leaves for depth 2", n_leaves <= 4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0, stratify=y)
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
acc = tree.score(X_te, y_te)
n_leaves = tree.get_n_leaves()

```

</details>

### Exercise 3 · Stretch · Find the best split

Write `best_threshold(x, y)`: try the midpoint between every pair of consecutive sorted values and return the threshold with the **lowest weighted Gini** of the two sides (reuse your `gini`).

In [ ]:
def best_threshold(x, y):
    pass   # TODO


In [ ]:
try:
    import numpy as np
    t = best_threshold(np.array([1, 2, 3, 10, 11, 12]), np.array([0, 0, 0, 1, 1, 1]))
    check("splits the two groups", 3 < t < 10)
    t2 = best_threshold(np.array([1, 2, 3, 4, 5, 6]), np.array([0, 0, 1, 1, 1, 1]))
    check("second example", t2 == 2.5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def best_threshold(x, y):
    x, y = np.asarray(x), np.asarray(y)
    order = np.argsort(x); xs, ys = x[order], y[order]
    best_t, best_score = None, np.inf
    for i in range(1, len(xs)):
        if xs[i] == xs[i - 1]:
            continue
        t = (xs[i] + xs[i - 1]) / 2
        left, right = ys[xs <= t], ys[xs > t]
        score = (len(left) * gini(left) + len(right) * gini(right)) / len(ys)
        if score < best_score:
            best_t, best_score = t, score
    return best_t

```

This exhaustive scan over features and thresholds is the entire core of CART training.

</details>

---
*Back to the course: **Machine Learning End To End → Decision Trees**.*